In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
ProjDIR = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/" # Change to your project directory
sys.path.insert(1, f'{ProjDIR}/src/')
from CellType_PSY import *


try:
    os.chdir(f"{ProjDIR}/dev_notebooks/")
    print(f"Current working directory: {os.getcwd()}")
except FileNotFoundError as e:
    print(f"Error: Could not change directory - {e}")
except Exception as e:  
    print(f"Unexpected error: {e}")    


import yaml
with open(ProjDIR + '/config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)

In [ ]:
#Bias_Save_Dir = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/results/random/Centering/"
Bias_Save_Dir = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/results/match/Centering/CDS_WB_LOEUF_Tricubic/"
ASD_All_Bias = pd.read_csv(Bias_Save_Dir + "ASD_All_bias_addP.csv", index_col=0)
SCZ_Bias = pd.read_csv(Bias_Save_Dir + "SCZ_bias_addP.csv", index_col=0)
HighIQ_ASD_Bias = pd.read_csv(Bias_Save_Dir + "ASD_HIQ_bias_addP.csv", index_col=0)
LowIQ_ASD_Bias = pd.read_csv(Bias_Save_Dir + "ASD_LIQ_bias_addP.csv", index_col=0)
X22q_Bias = pd.read_csv(Bias_Save_Dir + "22q_del_bias_addP.csv", index_col=0)

#if "match" in 
#DDD_Bias = pd.read_csv(Bias_Save_Dir + "DDD_61_bias_addP.csv", index_col=0)
DDD_Bias = pd.read_csv(Bias_Save_Dir + "DDD_bias_addP.csv", index_col=0)

# VNR_Pos_Bias = pd.read_csv(Bias_Save_Dir + "UKBB_VNR_Pos_bias_addP.csv", index_col=0)
# VNR_Neg_Bias = pd.read_csv(Bias_Save_Dir + "UKBB_VNR_Neg_bias_addP.csv", index_col=0)
# EDU_Pos_Bias = pd.read_csv(Bias_Save_Dir + "UKBB_EDU_Pos_bias_addP.csv", index_col=0)
# EDU_Neg_Bias = pd.read_csv(Bias_Save_Dir + "UKBB_EDU_Neg_bias_addP.csv", index_col=0)


In [ ]:
Bias_Save_Dir = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/results/random/Centering/"
#Bias_Save_Dir = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/results/match/Centering/CDS_WB_LOEUF_Tricubic/"
ASD_All_Bias = pd.read_csv(Bias_Save_Dir + "ASD_All_bias_addP.csv", index_col=0)
SCZ_Bias = pd.read_csv(Bias_Save_Dir + "SCZ_bias_addP.csv", index_col=0)
HighIQ_ASD_Bias = pd.read_csv(Bias_Save_Dir + "ASD_HIQ_bias_addP.csv", index_col=0)
LowIQ_ASD_Bias = pd.read_csv(Bias_Save_Dir + "ASD_LIQ_bias_addP.csv", index_col=0)
X22q_Bias = pd.read_csv(Bias_Save_Dir + "22q_del_bias_addP.csv", index_col=0)
DDD_Bias = pd.read_csv(Bias_Save_Dir + "DDD_61_bias_addP.csv", index_col=0)
#DDD_Bias = pd.read_csv(Bias_Save_Dir + "DDD_bias_addP.csv", index_col=0)

VNR_Pos_Bias = pd.read_csv(Bias_Save_Dir + "UKBB_VNR_Pos_bias_addP.csv", index_col=0)
VNR_Neg_Bias = pd.read_csv(Bias_Save_Dir + "UKBB_VNR_Neg_bias_addP.csv", index_col=0)
EDU_Pos_Bias = pd.read_csv(Bias_Save_Dir + "UKBB_EDU_Pos_bias_addP.csv", index_col=0)
EDU_Neg_Bias = pd.read_csv(Bias_Save_Dir + "UKBB_EDU_Neg_bias_addP.csv", index_col=0)


# ASD_vs_SCZ 

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

def sign_flip_permutation_test(x, y, n_perms=10000):
    """
    Performs a random sign flip permutation test for paired data.
    This is robust to cluster-dependence (pseudoreplication) because it 
    preserves the within-pair magnitude while randomizing the direction of effect.
    """
    # 1. Calculate the actual observed mean difference
    diffs = np.array(x) - np.array(y)
    obs_diff = np.mean(diffs)
    
    n_samples = len(diffs)
    
    # 2. Vectorized Permutation
    # Generate a matrix of random signs (+1 or -1)
    # Shape: (n_perms, n_samples)
    signs = np.random.randint(0, 2, size=(n_perms, n_samples)) * 2 - 1
    
    # 3. Calculate mean differences for all permutations simultaneously
    # Broadcast multiplication: diffs * signs
    perm_means = np.mean(diffs * signs, axis=1)
    
    # 4. Calculate P-value (Two-tailed)
    # Proportion of permutations where absolute random difference >= absolute observed difference
    p_val = np.mean(np.abs(perm_means) >= np.abs(obs_diff))
    
    return p_val

def compare_biases(bias1, bias2, name1="1", name2="2", efflabel="EFFECT", neurons=None, n_perms=10000):
    """
    Compare two bias datasets using Mann-Whitney (Standard), Wilcoxon (Paired), 
    and Permutation Sign Flip (Robust/Recommended).
    """
    # Ensure combineBias is available or merge appropriately
    # Assuming combineBias aligns the clusters (e.g. Cluster A in bias1 aligns with Cluster A in bias2)
    BiasDF_cb = combineBias(bias1, bias2, name1=name1, name2=name2)

    results_list = []
    
    # Determine which list of CTs to use
    # (Assuming ALL_CTs is a global variable from your context, otherwise using unique values)
    if neurons is not None:
        target_cts = neurons
    else:
        # Fallback if ALL_CTs isn't defined globally
        target_cts = BiasDF_cb[f"Supercluster_{name1}"].unique()

    print(f"Running analysis with {n_perms} permutations per cell type...")

    for CT in target_cts:
        # Filter data for current CT
        CT_mask = BiasDF_cb[f"Supercluster_{name1}"] == CT
        CT_data = BiasDF_cb[CT_mask]
        
        # Get bias values
        CT_Bias1 = CT_data[f"{efflabel}_{name1}"].values
        CT_Bias2 = CT_data[f"{efflabel}_{name2}"].values
        
        # Skip if empty or unequal lengths (paired tests require equal lengths)
        if len(CT_Bias1) == 0 or len(CT_Bias2) == 0:
            print(f"Warning: No data found for {CT}")
            continue
        if len(CT_Bias1) != len(CT_Bias2):
            print(f"Warning: Mismatched cluster counts for {CT}. Cannot run paired tests.")
            continue

        # Standard Error of Mean
        stm_1 = np.std(CT_Bias1) / np.sqrt(len(CT_Bias1))
        stm_2 = np.std(CT_Bias2) / np.sqrt(len(CT_Bias2))
            
        # Calculate Mean Difference
        bias_diff = np.mean(CT_Bias1) - np.mean(CT_Bias2)
        
        try:
            # 1. Mann-Whitney U (The "Standard" but flawed test for this context)
            t_man, p_man = stats.mannwhitneyu(CT_Bias1, CT_Bias2)
            
            # 2. Wilcoxon Signed-Rank (Standard paired test)
            # Add small noise or handle zeros if bias1 == bias2 to prevent errors
            if np.allclose(CT_Bias1, CT_Bias2):
                p_wil = 1.0
            else:
                t_wil, p_wil = stats.wilcoxon(CT_Bias1, CT_Bias2)
            
            # 3. Random Sign Flip Permutation (The "Robust" test)
            p_perm = sign_flip_permutation_test(CT_Bias1, CT_Bias2, n_perms=n_perms)
            
        except Exception as e:
            print(f"Error calculating statistics for {CT}: {str(e)}")
            continue
        
        results_list.append({
            'Supercluster': CT,
            'N_Clusters': len(CT_Bias1),  # Useful to see how N impacts P-value
            f'Bias_{name1}': np.mean(CT_Bias1),
            f'Bias_{name2}': np.mean(CT_Bias2), 
            f'STM_{name1}': stm_1,
            f'STM_{name2}': stm_2,
            'Bias_Diff': bias_diff,
            'Mann_Whitney_P': p_man,
            'Wilcoxon_P': p_wil,
            'Permutation_P': p_perm  # <--- New Robust Metric
        })

    # Convert to DataFrame
    results = pd.DataFrame(results_list)
    
    if len(results) > 0:
        results = results.sort_values(by="Bias_Diff")
        
        # Perform FDR correction on ALL P-values
        results['Mann_Whitney_FDR'] = multipletests(results['Mann_Whitney_P'], method='fdr_bh')[1]
        results['Wilcoxon_FDR'] = multipletests(results['Wilcoxon_P'], method='fdr_bh')[1]
        results['Permutation_FDR'] = multipletests(results['Permutation_P'], method='fdr_bh')[1]
        
        # Bonferroni correction (conservative)
        results['Bonferroni_P'] = (results['Permutation_P'] * results.shape[0]).clip(upper=1.0)
        
        results = results.set_index("Supercluster")
    else:
        print("Warning: No results generated")
        
    return results

In [ ]:
name1="ASD"
name2="SCZ"
EffLabel = "EFFECT"
ASD_SCZ_Contrast = compare_biases(HighIQ_ASD_Bias, SCZ_Bias, name1="ASD", name2="SCZ", efflabel=EffLabel)
ASD_SCZ_Contrast_Neurons = ASD_SCZ_Contrast[ASD_SCZ_Contrast.index.isin(Neurons)]
plot_bias_comparison(ASD_SCZ_Contrast_Neurons, name1, name2, p_test="Bonferroni_P", legend_anchor=(0.15, 0.9))

In [ ]:
ASD_SCZ_Contrast_Neurons

In [ ]:
EffLabel = "EFFECT"

CompareSingleCT(HighIQ_ASD_Bias, SCZ_Bias, "Medium spiny neuron",  ASD_SCZ_Contrast,
                     "ASD Mutation Bias", "SCZ Mutation Bias", efflabel=EffLabel, pval="Bonferroni_P", loc=(0.05, 0.23))
CompareSingleCT(HighIQ_ASD_Bias, SCZ_Bias, "MGE interneuron",   ASD_SCZ_Contrast,
                    "ASD Mutation Bias", "SCZ Mutation Bias", loc=(0.1, 0.05))

## compared with random genes

In [ ]:
ASD_Null = pd.read_csv("{}/null_bias/ASD_HIQ_null_bias.csv".format(Bias_Save_Dir), index_col=0)
ASD_Null.columns = [int(col) for col in ASD_Null.columns]
SCZ_Null = pd.read_csv("{}/null_bias/SCZ_null_bias.csv".format(Bias_Save_Dir), index_col=0)
SCZ_Null.columns = [int(col) for col in SCZ_Null.columns]

In [ ]:
ASD_Null.head(2)

In [ ]:
#Supercluster2Test = "MGE interneuron"
Supercluster2Test = "CGE interneuron"
Supercluster_Idx = Anno[Anno["Supercluster"] == Supercluster2Test].index
obs_asd_mean = HighIQ_ASD_Bias.loc[Supercluster_Idx, "EFFECT"].mean()
obs_scz_mean = SCZ_Bias.loc[Supercluster_Idx, "EFFECT"].mean()
obs = obs_asd_mean - obs_scz_mean
print(obs_asd_mean, obs_scz_mean, obs_asd_mean - obs_scz_mean)

In [ ]:
Supercluster_Idx

In [ ]:
bias_diffs = []
bias_gt_obs = 0
for i in range(10000):
    bias_ASD_Null = ASD_Null.loc[Supercluster_Idx, i]
    bias_SCZ_Null = SCZ_Null.loc[Supercluster_Idx, i]
    bias_ASD_Null_mean = bias_ASD_Null.mean()
    bias_SCZ_Null_mean = bias_SCZ_Null.mean()
    bias_diff = bias_ASD_Null.mean() - bias_SCZ_Null.mean()
    bias_diffs.append(bias_diff)
    #if bias_ASD_Null_mean < obs_asd_mean and bias_SCZ_Null_mean > obs_scz_mean:
    if bias_SCZ_Null_mean > obs_scz_mean and abs(bias_diff) > abs(obs):
        bias_gt_obs += 1
print(bias_gt_obs/10000)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

obs = obs_asd_mean - obs_scz_mean

# Calculate one-tailed p-value: P(null > obs)
bias_diffs_arr = np.array(bias_diffs)
pval = np.mean(bias_diffs_arr > obs)

plt.figure(figsize=(7, 5))
plt.hist(bias_diffs, bins=10, color='lightgray', edgecolor='black', alpha=0.7)
plt.axvline(obs, color='red', linestyle='--', linewidth=2, label=f'Observed (obs={obs})')
plt.xlabel("Null Difference in Bias")
plt.ylabel("Count")
plt.title(f"Observed Bias Difference vs Null Distribution\nOne-tailed p-value (obs < null): {pval:.4g}")
plt.legend()
plt.show()

In [ ]:
Supercluster2Test = "Medium spiny neuron"
Supercluster_Idx = Anno[Anno["Supercluster"] == Supercluster2Test].index
obs_asd_mean = HighIQ_ASD_Bias.loc[Supercluster_Idx, "EFFECT"].mean()
obs_scz_mean = SCZ_Bias.loc[Supercluster_Idx, "EFFECT"].mean()
print(obs_asd_mean, obs_scz_mean, obs_asd_mean - obs_scz_mean)

In [ ]:
bias_diffs = []
bias_gt_obs = 0
for i in range(10000):
    bias_ASD_Null = ASD_Null.loc[Supercluster_Idx, i]
    bias_SCZ_Null = SCZ_Null.loc[Supercluster_Idx, i]
    bias_ASD_Null_mean = bias_ASD_Null.mean()
    bias_SCZ_Null_mean = bias_SCZ_Null.mean()
    bias_diff = bias_ASD_Null.mean() - bias_SCZ_Null.mean()
    bias_diffs.append(bias_diff)
    if bias_ASD_Null_mean < obs_asd_mean and bias_SCZ_Null_mean > obs_scz_mean:
        bias_gt_obs += 1
print(bias_gt_obs/10000)


In [ ]:
# Calculate one-tailed p-value: P(null > obs)
bias_diffs_arr = np.array(bias_diffs)
pval = np.mean(bias_diffs_arr > obs)

plt.figure(figsize=(7, 5))
plt.hist(bias_diffs, bins=10, color='lightgray', edgecolor='black', alpha=0.7)
plt.axvline(obs, color='red', linestyle='--', linewidth=2, label=f'Observed (obs={obs})')
plt.xlabel("Null Difference in Bias")
plt.ylabel("Count")
plt.title(f"Observed Bias Difference vs Null Distribution\nOne-tailed p-value (obs < null): {pval:.4g}")
plt.legend()
plt.show()

In [ ]:
## 22q11.2

In [ ]:
VIP_Anno = pd.read_csv(f"{ProjDIR}/notebooks/VIP_Anno.csv", index_col=0)

In [ ]:
VIP_Pos = VIP_Anno[VIP_Anno["VIP"] > 1].index.values
VIP_Neg = VIP_Anno[VIP_Anno["VIP"] < 1].index.values

In [ ]:
X22q_vip_pos_mean = X22q_Bias.loc[VIP_Pos, "EFFECT"].mean()
X22q_vip_neg_mean = X22q_Bias.loc[VIP_Neg, "EFFECT"].mean()
#X22q_vip_pos_mean - X22q_vip_neg_mean
obs = X22q_vip_pos_mean - X22q_vip_neg_mean
obs

In [ ]:
X22q_Null = pd.read_csv("{}/null_bias/22q_del_null_bias.csv".format(Bias_Save_Dir), index_col=0)
X22q_Null.columns = [int(col) for col in X22q_Null.columns]

In [ ]:
bias_diffs = []
bias_gt_obs = 0
for i in range(10000):
    rand_X22q_vip_pos_mean = X22q_Null.loc[VIP_Pos, i].mean()
    rand_X22q_vip_neg_mean = X22q_Null.loc[VIP_Neg, i].mean()
    rand_bias_diff = rand_X22q_vip_pos_mean - rand_X22q_vip_neg_mean
    bias_diffs.append(rand_bias_diff)
    #if rand_X22q_vip_pos_mean > X22q_vip_pos_mean and rand_X22q_vip_neg_mean < X22q_vip_neg_mean:
    if rand_X22q_vip_pos_mean > X22q_vip_pos_mean and rand_X22q_vip_pos_mean-rand_X22q_vip_neg_mean > obs:
        bias_gt_obs += 1
print(bias_gt_obs/10000)


In [ ]:
# Calculate one-tailed p-value: P(null > obs)
obs = X22q_vip_pos_mean - X22q_vip_neg_mean
bias_diffs_arr = np.array(bias_diffs)
pval = np.mean(bias_diffs_arr > obs)

plt.figure(figsize=(7, 5))
plt.hist(bias_diffs, bins=10, color='lightgray', edgecolor='black', alpha=0.7)
plt.axvline(obs, color='red', linestyle='--', linewidth=2, label=f'Observed (obs={obs})')
plt.xlabel("Null Difference in Bias")
plt.ylabel("Count")
plt.title(f"Observed Bias Difference vs Null Distribution\nOne-tailed p-value (obs < null): {pval:.4g}")
plt.legend()
plt.show()